# Experiment 2 — Context Ladder for Real-Bug Detection with a Code LLM

**PhD topic:** Large Language Models for Automated Program Debugging · **Builds on:** Experiment 1 (`BugInPy_CodeBERT_Experiment.ipynb`, results in `RESULTS_experiment1.md`) · **Plan:** `EXPERIMENT2_PLAN.md`

Experiment 1 showed that fine-tuned CodeBERT detects real BugInPy bugs at chance level from the function text alone (ROC-AUC ≈ 0.50), while the same pipeline detects injected obvious bugs perfectly. Experiment 2 asks **which context a code LLM needs to detect real bugs**, and **whether its successes come from understanding or from memorising BugInPy**.

| RQ | Question |
|---|---|
| RQ1 | Does a code LLM, zero-shot, detect real bugs from the function alone (C0) better than chance and than CodeBERT? |
| RQ2 | Which context makes real bugs detectable: specification (C1), file skeleton (C2), failing test (C3)? |
| RQ3 | Is success due to memorisation (identifier-renaming probe)? |
| RQ4 | Sanity: does the LLM pipeline detect injected obvious bugs (positive control)? |

## How to run (lessons from Experiment 1)
1. Use a **GPU runtime (T4)**. Check that the bottom bar shows `T4`.
2. In a **new session**, run **E2-1.1 (installation) FIRST**. If Colab asks to *Restart session*, accept, then continue with E2-1.2. Do **not** run E2-1.1 again in the same session.
3. Then run the cells **in order, one by one**.
4. Everything is saved on Drive under `PhD_Experiments/BugInPy_CodeBERT/exp2/`.

| Phase | Content | Status |
|---|---|---|
| E2-1 | Setup, model loading, scoring function, smoke test | done |
| E2-2 | Evaluation subset, context ladder C0–C3, leakage controls, renaming | **available** |
| E2-3 | Prompt selection on the dev set | pending |
| E2-4 | Positive control | pending |
| E2-5 | Inference (resumable) | pending |
| E2-6 | Evaluation | pending |
| E2-7 | Analysis & report | pending |

---
# PHASE E2-1 — Setup, model and scoring function

**Goal:** a reproducible environment, the code LLM loaded in 4-bit on a T4, and a **deterministic scoring function** that turns the LLM's answer into a probability.

**Scoring principle.** The prompt ends with *"Does this function contain a bug? Answer with a single word: Yes or No."*. Instead of letting the model write an answer, we read the probabilities of the next token:

> score = P(Yes) / (P(Yes) + P(No))

This needs **one forward pass** (fast), involves **no sampling** (deterministic), and gives a probability, so ROC-AUC and paired comparisons can be computed exactly as in Experiment 1. We also record `mass_yes_no` = P(Yes) + P(No). If it is low, the model wanted to answer something other than Yes/No, and the prompt must be improved (Phase E2-3).

## E2-1.1 Installation (run FIRST in every new session)

**Explanation.** Installs `bitsandbytes` (4-bit quantisation, needed to fit a 7-billion-parameter model on a 15 GB T4), and makes sure that `transformers` and `accelerate` are compatible.

If packages were changed, the cell **restarts the Python session automatically**. Colab then shows *"Your session crashed/restarted"*, which is **expected**. Mixing an already imported old `transformers` with newly installed files causes errors such as `No module named 'transformers.models.…'`.

**Expected output.** pip messages, the versions before/after, then either the automatic restart (continue with **E2-1.2**) or `[OK] E2-1.1`.


In [ ]:
# ---- E2-1.1 Installation (restarts the session automatically when needed) ----
import importlib.metadata as _md, sys as _sys
def _v(p):
    try: return _md.version(p)
    except _md.PackageNotFoundError: return None
_before = {p: _v(p) for p in ("transformers", "accelerate", "bitsandbytes")}
%pip install -q -U "transformers>=4.44,<5" "accelerate>=0.33" "bitsandbytes>=0.43" "scikit-learn>=1.3" tqdm
_after = {p: _v(p) for p in ("transformers", "accelerate", "bitsandbytes")}
print("before:", _before); print("after :", _after)
if _before != _after or "transformers" in _sys.modules:
    print("\nPackages changed (or transformers was already imported): RESTARTING the session automatically.")
    print("When the session has restarted, continue with E2-1.2 (do NOT run E2-1.1 again).")
    import os as _os; _os.kill(_os.getpid(), 9)        # standard Colab way to restart the Python process
print("\n[OK] E2-1.1 — packages already up to date, continue with E2-1.2.")


## E2-1.2 Environment, Drive and experiment configuration

**Explanation.** Imports, seed, GPU information, Google Drive, and the paths of **Experiment 1** (whose data are reused) and **Experiment 2** (new outputs in `exp2/`). The cell checks that the Experiment 1 artefacts needed later are on Drive.

`RESUME_RUN_ID`: leave it `None` the first time. After a restart, put the printed run ID here so that everything is saved under the same run.

**Expected output.** Python and GPU information (`Tesla T4`), the Drive paths, `Experiment 1 artefacts: all found`, the run ID, and `[OK] E2-1.2`.

In [ ]:
# ---- E2-1.2 Environment & configuration ----
import os, sys, platform, json, time, random, re, ast, hashlib, datetime, gc, textwrap, warnings, subprocess
from pathlib import Path
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np, pandas as pd
import torch, transformers
import importlib.metadata as _md
if transformers.__version__ != _md.version("transformers"):
    raise RuntimeError(f"transformers in memory ({transformers.__version__}) differs from the installed one "
                       f"({_md.version('transformers')}): use Runtime > Restart session, then run E2-1.2 again.")
from tqdm.auto import tqdm
warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

IN_COLAB = "google.colab" in sys.modules
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python      :", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__)
print("Device      :", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU")
if DEVICE.type == "cuda":
    print(f"GPU memory  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

USE_DRIVE = True
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/PhD_Experiments/BugInPy_CodeBERT")
else:
    BASE_DIR = Path.cwd() / "BugInPy_CodeBERT"
EXP1 = {"processed": BASE_DIR / "data" / "processed", "interim": BASE_DIR / "data" / "interim",
        "raw": BASE_DIR / "data" / "raw", "metrics": BASE_DIR / "results" / "metrics",
        "predictions": BASE_DIR / "results" / "predictions"}
E2 = {k: BASE_DIR / "exp2" / k for k in ("data", "cache", "results", "figures", "logs")}
for p in E2.values(): p.mkdir(parents=True, exist_ok=True)
WORK = (Path("/content") if IN_COLAB else Path.cwd()) / "work_exp2"; WORK.mkdir(parents=True, exist_ok=True)

needed = {"phase5 dataset": EXP1["processed"] / "phase5_dataset_v2.jsonl",
          "phase6 splits": EXP1["processed"] / "phase6_splits.csv",
          "phase4 source files": EXP1["interim"] / "phase4_source_files.tar.gz",
          "BugInPy archive": next(iter(sorted(EXP1["raw"].glob("BugsInPy_*.tar.gz"))), EXP1["raw"] / "BugsInPy_MISSING.tar.gz"),
          "CodeBERT OOF (extended)": EXP1["predictions"] / "phase8_extended_oof_predictions.csv"}
missing = [k for k, p in needed.items() if not p.exists()]
print("\nExperiment 1 artefacts:", "all found" if not missing else f"MISSING {missing}")

RESUME_RUN_ID = None
RUN_ID = RESUME_RUN_ID or "exp2_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
cfg_path = E2["logs"] / f"config_{RUN_ID}.json"
CONFIG2 = json.loads(cfg_path.read_text()) if cfg_path.exists() else {}
CONFIG2.update({"run_id": RUN_ID, "seed": SEED, "base_dir": str(BASE_DIR),
                "hardware": {"device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None},
                "libraries": {"torch": torch.__version__, "transformers": transformers.__version__, "python": sys.version.split()[0]}})
def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=str)); return Path(path)
save_json(CONFIG2, cfg_path)
print("RUN_ID      :", RUN_ID, "-> keep it for RESUME_RUN_ID after a restart")
print("Exp. 2 dir  :", BASE_DIR / "exp2")

# ---- Verification ----
assert not missing, f"Experiment 1 files missing on Drive: {missing}"
if DEVICE.type != "cuda":
    print("WARNING: no GPU. Switch to a T4 runtime before E2-1.3 (Runtime > Change runtime type).")
print("\n[OK] E2-1.2")

## E2-1.3 Load the code LLM (4-bit)

**Explanation.** Loads **Qwen2.5-Coder-7B-Instruct** (open weights, instruction-tuned for code) in **4-bit NF4** quantisation with `bitsandbytes`, so that it fits in about 5–6 GB of GPU memory. The first time, about 15 GB are downloaded from Hugging Face (≈ 3–6 minutes). The exact model revision (commit hash) is recorded for reproducibility.

`SMOKE_TEST = True` loads a tiny 0.5B model instead. It is used only to check that the code runs, never for results.

**Expected output.** The model name and revision, its memory footprint (≈ 5–6 GB), and `[OK] E2-1.3`.

In [ ]:
# ---- E2-1.3 Load the LLM ----
_missing = [n for n in ("DEVICE", "CONFIG2", "save_json", "cfg_path") if n not in globals()]
if _missing:
    raise RuntimeError("Run E2-1.2 first in this session (the session was restarted, e.g. after changing the runtime type). "
                       f"Missing: {_missing}")
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
SMOKE_TEST = False
MODEL_ID = "Qwen/Qwen2.5-Coder-0.5B-Instruct" if SMOKE_TEST else "Qwen/Qwen2.5-Coder-7B-Instruct"
if not SMOKE_TEST and DEVICE.type != "cuda":
    raise RuntimeError("A T4 GPU is required for the 7B model: Runtime > Change runtime type > T4 GPU, then re-run E2-1.2 and this cell.")
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if DEVICE.type == "cuda" and not SMOKE_TEST:
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=qcfg, device_map="auto", torch_dtype=torch.float16)
else:
    llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
llm.eval()
MODEL_REVISION = getattr(llm.config, "_commit_hash", None)
mem = llm.get_memory_footprint() / 1024**3
print(f"Model      : {MODEL_ID}")
print(f"Revision   : {MODEL_REVISION}")
print(f"Footprint  : {mem:.2f} GB | load time {time.time() - t0:.0f}s | context window {llm.config.max_position_embeddings} tokens")
CONFIG2["model"] = {"id": MODEL_ID, "revision": MODEL_REVISION, "quantization": "4-bit NF4 (bitsandbytes)" if DEVICE.type == "cuda" and not SMOKE_TEST else "none",
                    "smoke_test": SMOKE_TEST}
save_json(CONFIG2, cfg_path)

# ---- Verification ----
assert llm.config.max_position_embeddings >= 8192
print("\n[OK] E2-1.3" + ("  (SMOKE TEST model — not for results)" if SMOKE_TEST else ""))

## E2-1.4 Scoring function

**Explanation.** `bug_score(code, context=None)` builds a chat prompt with the model's own template:
* **system:** *"You are an expert Python code reviewer."*
* **user:** optional context sections (filled in Phase E2-2), then the function in a ```python block, then the question.

It then reads the next-token distribution and returns:
* `score` = P(Yes) / (P(Yes) + P(No)), the **bug score**;
* `mass_yes_no` = P(Yes) + P(No), which measures how much the model "wants" to answer Yes/No;
* `n_tokens`, the prompt length.

The Yes/No probabilities sum over the usual spellings ("Yes", "yes", " Yes", …).

**Expected output.** The token ids used for Yes and No, and `[OK] E2-1.4`.

In [ ]:
# ---- E2-1.4 Scoring function ----
_missing = [n for n in ("DEVICE", "CONFIG2", "llm", "tok") if n not in globals()]
if _missing:
    raise RuntimeError("Run E2-1.2 and E2-1.3 first in this session (the session was restarted, e.g. after changing the runtime type). "
                       f"Missing: {_missing}")
SYSTEM_PROMPT = "You are an expert Python code reviewer."
QUESTION = "Does this function contain a bug? Answer with a single word: Yes or No."

def first_ids(words):
    return sorted({tok.encode(w, add_special_tokens=False)[0] for w in words})
YES_IDS = first_ids(["Yes", "yes", " Yes", " yes", "YES"])
NO_IDS  = first_ids(["No", "no", " No", " no", "NO"])
assert not set(YES_IDS) & set(NO_IDS)

def build_user_text(code, context=None, question=QUESTION):
    parts = []
    for title, body in (context or []):
        parts.append(f"### {title}\n{body}")
    parts.append(f"### Function to review\n```python\n{code}\n```")
    parts.append(question)
    return "\n\n".join(parts)

def build_prompt(user_text, system=SYSTEM_PROMPT):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user_text}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def bug_score(code, context=None, question=QUESTION, system=SYSTEM_PROMPT):
    enc = tok(build_prompt(build_user_text(code, context, question), system), return_tensors="pt").to(llm.device)
    logits = llm(**enc).logits[0, -1].float()
    probs = torch.softmax(logits, dim=-1)
    p_yes, p_no = probs[YES_IDS].sum().item(), probs[NO_IDS].sum().item()
    return {"score": p_yes / (p_yes + p_no) if p_yes + p_no > 0 else 0.5,
            "p_yes": p_yes, "p_no": p_no, "mass_yes_no": p_yes + p_no, "n_tokens": int(enc["input_ids"].shape[1])}

print("Yes token ids:", YES_IDS, [tok.decode([i]) for i in YES_IDS])
print("No  token ids:", NO_IDS, [tok.decode([i]) for i in NO_IDS])
print("\n[OK] E2-1.4")

## E2-1.5 Smoke test

**Explanation.** Four tiny hand-written functions, where the answer is obvious to a human:

| # | Function | Expected |
|---|---|---|
| 1 | correct `add(a, b)` | low score |
| 2 | `add` with an injected `1 / 0` | high score |
| 3 | `sum_list` that skips the last element (`range(len(xs) - 1)`) | higher than #4 |
| 4 | correct `sum_list` | low score |

We also check that the score is **deterministic** (same input → same score), that the model answers Yes/No (`mass_yes_no` high), and how long one prompt takes.

**Expected output.** A table with the four scores; for the 7B model, #2 and #3 should get clearly higher scores than #1 and #4. Then the time per prompt (≈ 0.2–1 s on a T4 for short prompts) and `[OK] E2-1.5`.

*This is not an evaluation*, only a check that the scoring behaves sensibly.

In [ ]:
# ---- E2-1.5 Smoke test ----
_missing = [n for n in ("DEVICE", "CONFIG2", "llm", "tok", "bug_score") if n not in globals()]
if _missing:
    raise RuntimeError("Run E2-1.2, E2-1.3 and E2-1.4 first in this session (the session was restarted, e.g. after changing the runtime type). "
                       f"Missing: {_missing}")
examples = {
    "1 correct add":        "def add(a, b):\n    return a + b",
    "2 add + injected 1/0": "def add(a, b):\n    _ = 1 / 0\n    return a + b",
    "3 sum_list off-by-one":"def sum_list(xs):\n    total = 0\n    for i in range(len(xs) - 1):\n        total += xs[i]\n    return total",
    "4 correct sum_list":   "def sum_list(xs):\n    total = 0\n    for i in range(len(xs)):\n        total += xs[i]\n    return total",
}
rows, t0 = [], time.time()
for name, code in examples.items():
    r = bug_score(code); rows.append({"example": name, **r})
dt = (time.time() - t0) / len(examples)
smoke = pd.DataFrame(rows).set_index("example")
display(smoke[["score", "p_yes", "p_no", "mass_yes_no", "n_tokens"]].round(4))
again = bug_score(examples["3 sum_list off-by-one"])["score"]
print(f"Time per prompt: {dt:.2f}s | deterministic: {abs(again - smoke.loc['3 sum_list off-by-one', 'score']) < 1e-4}")
ordering_ok = smoke.loc["2 add + injected 1/0", "score"] > smoke.loc["1 correct add", "score"] and \
              smoke.loc["3 sum_list off-by-one", "score"] > smoke.loc["4 correct sum_list", "score"]
print("Buggy examples scored higher than their correct versions:", ordering_ok)

CONFIG2["scoring"] = {"system": SYSTEM_PROMPT, "question": QUESTION, "yes_ids": YES_IDS, "no_ids": NO_IDS,
                      "smoke_test": smoke.round(4).reset_index().to_dict("records"), "seconds_per_short_prompt": round(dt, 3)}
save_json(CONFIG2, cfg_path)

# ---- Verification ----
assert abs(again - smoke.loc["3 sum_list off-by-one", "score"]) < 1e-4, "scoring is not deterministic"
assert smoke.mass_yes_no.min() > 0.5 or SMOKE_TEST, "the model does not answer Yes/No — prompt problem"
print("\n[OK] PHASE E2-1 COMPLETE" + (" (SMOKE TEST)" if SMOKE_TEST else ""))

**Please report back** the outputs of **E2-1.2** (GPU, artefacts, RUN_ID), **E2-1.3** (model, revision, footprint) and **E2-1.5** (the score table, time per prompt, ordering).

➡️ Continue with **PHASE E2-2** below.

---
# PHASE E2-2 — Evaluation subset and context ladder

> **After a restart:** E2-1.1 (if needed) → E2-1.2 → E2-1.3 → E2-1.4, then E2-2.1 → E2-2.5. E2-2 needs the **tokenizer** (to count tokens) and **internet** (to fetch the failing tests from GitHub). It runs in ≈ 5 minutes and does not use the GPU much.

**Goal:** build every prompt of the experiment once, with its context, leakage controls and token budget. The prompts are saved to `exp2/data/ladder_items.jsonl`, so that E2-3 … E2-5 only have to score them.

| Section | Content |
|---|---|
| E2-2.1 | Pairs, **dev set** (one held-out project), **evaluation subset** (one pair per bug), unchanged functions, positive-control functions |
| E2-2.2 | Context C1 (docstring) and C2 (file skeleton) from the **buggy** commit |
| E2-2.3 | Context C3: the **bug-revealing test**, fetched from the original GitHub repository |
| E2-2.4 | **Renaming** transformation (memorisation probe) and its verification |
| E2-2.5 | Assemble all prompts with token budgets, statistics, and saving |

**Design decisions (from `EXPERIMENT2_PLAN.md`), adapted after E2-1:**
* Scoring speed on the T4 ≈ 2 s per short prompt, so the evaluation subset is **150 buggy/fixed pairs + 75 unchanged functions** (one pair per bug, random with seed 42). It can be enlarged later.
* **Dev set** = all pairs of one project (`luigi`), used **only** for prompt selection (E2-3) and **excluded** from the evaluation.
* Neutral wording of the test section (*"Test case from the project's test suite related to this code"*). We do **not** tell the model that the test fails, because that statement would be false for the fixed version.

## E2-2.1 Pairs, dev set, evaluation subset, positive-control set

**Explanation.**
* A **pair** = the buggy version of a function and its fixed version (same bug, file and function), taken from the Phase 5 dataset of Experiment 1.
* To keep the samples independent, we keep **one pair per bug** (chosen at random, seed 42).
* **Dev set:** the pairs of `luigi`. **Evaluation pool:** all other projects.
* **Evaluation subset:** 150 pairs and 75 unchanged functions (one per bug), drawn at random from the pool.
* **Positive control:** 100 clean functions from the pool. 50 of them, chosen at random, will receive the injected `_ = 1 / 0` (E2-2.5).

**Expected output.** Counts of pairs, dev pairs, evaluation items and positive-control items, and the number of evaluation pairs per project.

In [ ]:
# ---- E2-2.1 Subsets ----
_missing = [n for n in ("EXP1", "E2", "CONFIG2", "tok", "build_prompt", "build_user_text") if n not in globals()]
if _missing: raise RuntimeError(f"Run E2-1.2, E2-1.3 and E2-1.4 first. Missing: {_missing}")
import tarfile, copy, difflib, shutil
d5 = pd.read_json(EXP1["processed"] / "phase5_dataset_v2.jsonl", lines=True, dtype={"commit_sha": str})
b = d5[d5.label_source == "buggy_modified"]; f = d5[d5.label_source == "fixed_counterpart"]
pairs_all = b.merge(f, on=["sample_bug_key", "file_path", "qualname"], suffixes=("_b", "_f"))
pairs_all = pairs_all.rename(columns={"project_b": "project", "bug_id_b": "bug_id"})
pairs_1 = (pairs_all.sample(frac=1, random_state=SEED).drop_duplicates("sample_bug_key")
           .sort_values("sample_id_b").reset_index(drop=True))
pairs_1["pair_id"] = pairs_1.sample_bug_key + "::" + pairs_1.qualname

DEV_PROJECT, N_PAIRS, N_UNCH, N_PC = "luigi", 150, 75, 100
dev_pairs = pairs_1[pairs_1.project == DEV_PROJECT].reset_index(drop=True)
pool = pairs_1[pairs_1.project != DEV_PROJECT]
eval_pairs = pool.sample(n=min(N_PAIRS, len(pool)), random_state=SEED).sort_values("pair_id").reset_index(drop=True)
unch_pool = d5[(d5.label_source == "unchanged_same_file") & (d5.project != DEV_PROJECT)].drop_duplicates("sample_bug_key")
eval_unch = unch_pool.sample(n=min(N_UNCH, len(unch_pool)), random_state=SEED).sort_values("sample_id").reset_index(drop=True)
clean_pool = d5[(d5.label == 0) & (d5.project != DEV_PROJECT)]
pc_set = clean_pool.sample(n=min(N_PC, len(clean_pool)), random_state=SEED + 1).sort_values("sample_id").reset_index(drop=True)
pc_set["pc_label"] = (np.random.default_rng(SEED).random(len(pc_set)) < 0.5).astype(int)

print(f"Pairs (all) {len(pairs_all)} | one per bug {len(pairs_1)} | dev pairs ({DEV_PROJECT}) {len(dev_pairs)}")
print(f"Evaluation: {len(eval_pairs)} pairs ({2 * len(eval_pairs)} functions) + {len(eval_unch)} unchanged | "
      f"positive control: {len(pc_set)} ({pc_set.pc_label.sum()} to inject)")
print("Evaluation pairs per project:"); print(eval_pairs.project.value_counts().to_string())

# ---- Verification ----
assert not set(eval_pairs.project) & {DEV_PROJECT} and eval_pairs.sample_bug_key.is_unique
assert (eval_pairs.model_input_b != eval_pairs.model_input_f).all()
print("\n[OK] E2-2.1")

## E2-2.2 Context C1 (docstring) and C2 (file skeleton)

**Explanation.**
* **C1, specification:** the function's original **docstring**. Experiment 1 removed docstrings from the model input, so here they are added back explicitly. For a pair, the docstring of the **buggy** version is used for **both** members, because a fix sometimes edits the docstring, and that edit would reveal the fixed version.
* **C2, file skeleton:** the **imports**, the **class headers** and the **signatures** (no bodies) of the other functions of the same file, taken from the **buggy commit** for every member of a pair. A fix can add new helper functions to the file, and taking the skeleton from the fixed commit would reveal it. The function under review is replaced by a marker. The enclosing class comes first, so that it survives truncation.

The source files are restored from Experiment 1's archive `data/interim/phase4_source_files.tar.gz`.

**Expected output.** Coverage (share of evaluation functions with a docstring, and with a skeleton), and an example skeleton.

In [ ]:
# ---- E2-2.2 Docstrings & file skeletons ----
SRC_ROOT = WORK / "src_cache"
if not SRC_ROOT.exists():
    with tarfile.open(EXP1["interim"] / "phase4_source_files.tar.gz") as t: t.extractall(WORK, filter="data")
def read_source(project, sha, path):
    p = SRC_ROOT / project / sha[:12] / path
    return p.read_text(encoding="utf-8", errors="replace").replace("\r\n", "\n") if p.exists() else None

def parse_function(code):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try: tree = ast.parse(code)
        except SyntaxError: tree = ast.parse("class _W:\n" + code).body[0]
    return next(n for n in tree.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)))

def docstring_of(code_raw):
    try: return ast.get_docstring(parse_function(code_raw))
    except Exception: return None

def skeleton(source, target_qualname):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore"); tree = ast.parse(source)
    target = target_qualname.split("#")[0]
    imports = [ast.unparse(n) for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom))]
    def sig(fn, indent):
        g = copy.deepcopy(fn); g.body = [ast.Expr(ast.Constant(Ellipsis))]
        return textwrap.indent(ast.unparse(g), indent)
    def walk(body, prefix, indent, out):
        for n in body:
            if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)):
                out.append(indent + "# <function under review>" if prefix + n.name == target else sig(n, indent))
            elif isinstance(n, ast.ClassDef):
                bases = ", ".join(ast.unparse(x) for x in n.bases)
                out.append(f"{indent}class {n.name}" + (f"({bases})" if bases else "") + ":")
                walk(n.body, prefix + n.name + ".", indent + "    ", out)
    groups, contains = [], []
    for n in tree.body:
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            out = []; walk([n], "", "", out); groups.append("\n".join(out))
            contains.append(target == n.name or target.startswith(n.name + "."))
    ordered = [g for g, c in zip(groups, contains) if c] + [g for g, c in zip(groups, contains) if not c]
    return "\n".join(imports[:40]) + "\n\n" + "\n\n".join(ordered)

def contexts_for(project, sha_buggy, file_path, qualname, code_raw_for_doc):
    src_txt = read_source(project, sha_buggy, file_path)
    skel = None
    if src_txt is not None:
        try: skel = skeleton(src_txt, qualname)
        except Exception: skel = None
    return docstring_of(code_raw_for_doc), skel

eval_pairs[["doc", "skel"]] = [contexts_for(r.project, r.commit_sha_b, r.file_path, r.qualname, r.code_raw_b)
                               for r in eval_pairs.itertuples()]
dev_pairs[["doc", "skel"]] = [contexts_for(r.project, r.commit_sha_b, r.file_path, r.qualname, r.code_raw_b)
                              for r in dev_pairs.itertuples()]
eval_unch[["doc", "skel"]] = [contexts_for(r.project, r.commit_sha, r.file_path, r.qualname, r.code_raw)
                              for r in eval_unch.itertuples()]
cov = pd.DataFrame({"with docstring (%)": [100 * eval_pairs.doc.map(lambda x: isinstance(x, str)).mean(), 100 * eval_unch.doc.map(lambda x: isinstance(x, str)).mean()],
                    "with skeleton (%)": [100 * eval_pairs.skel.notna().mean(), 100 * eval_unch.skel.notna().mean()]},
                   index=["evaluation pairs", "unchanged"]).round(1)
display(cov)
ex = eval_pairs[eval_pairs.skel.notna()].iloc[0]
print(f"Example skeleton for {ex.pair_id} (first 18 lines):")
print("\n".join(ex.skel.split("\n")[:18]))

# ---- Verification ----
assert eval_pairs.skel.notna().mean() > 0.8, "too many missing file skeletons"
print("\n[OK] E2-2.2")

## E2-2.3 Context C3: the bug-revealing test

**Explanation.** For each bug, BugInPy's `run_test.sh` names the test(s) that fail on the buggy version and pass on the fixed one (pytest `file::Class::test`, or unittest `module.Class.test`). The test usually lives in the **fixed** commit, because developers add it together with the fix. We therefore:
1. parse the test identifiers from `run_test.sh`;
2. create partial clones of the original repositories (as in Exp. 1, Phase 4) and fetch the test file at the **fixed** commit;
3. extract the source of the named test function with `ast`.

The **same** test text is given to both members of a pair (symmetric). For an unchanged function, the test of its bug concerns *another* function; this case is analysed separately in E2-6.

**Expected output.** The share of bugs for which a test function could be extracted (expected > 80%), and an example. Duration ≈ 1–3 minutes (network).

In [ ]:
# ---- E2-2.3 Failing tests ----
BIP = WORK / "BugsInPy"
if not BIP.exists():
    arch = sorted(EXP1["raw"].glob("BugsInPy_*.tar.gz"))[-1]
    with tarfile.open(arch) as t: t.extractall(WORK, filter="data")
KV = re.compile(r'(\w+)\s*=\s*"([^"]*)"')
def bug_dir(project, bug_id): return BIP / "projects" / project / "bugs" / str(int(bug_id))
def info(p): return dict(KV.findall(p.read_text(errors="replace"))) if p.exists() else {}

def test_ids(project, bug_id):
    ids = []
    sh = bug_dir(project, bug_id) / "run_test.sh"
    for line in (sh.read_text(errors="replace").splitlines() if sh.exists() else []):
        for tokn in line.split():
            if "::" in tokn:                                    # pytest node id
                parts = tokn.split("::"); ids.append((parts[0], parts[1:-1], parts[-1]))
            elif re.fullmatch(r"[A-Za-z_][\w]*(\.[A-Za-z_][\w]*){2,}", tokn) and ("test" in tokn.lower()):
                ids.append((tokn, None, None))                  # unittest dotted id (resolved later)
    return ids[:3]

REPOS = WORK / "repos"; REPOS.mkdir(exist_ok=True)
def git(repo, *args, check=False, text=True):
    return subprocess.run(["git", "-C", str(repo), *args], capture_output=True, text=text)
def ensure_clone(project):
    dest = REPOS / project
    if not (dest / ".git").exists():
        url = info(BIP / "projects" / project / "project.info").get("github_url", "").rstrip("/")
        subprocess.run(["git", "clone", "-q", "--filter=blob:none", "--no-checkout", url, str(dest)], capture_output=True)
    return dest
def file_at(project, sha, path):
    repo = ensure_clone(project)
    r = git(repo, "show", f"{sha}:{path}", text=False)
    return r.stdout.decode("utf-8", "replace") if r.returncode == 0 else None

def extract_test(source, cls_parts, name):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore"); tree = ast.parse(source)
    scope = tree.body
    for c in (cls_parts or []):
        node = next((n for n in scope if isinstance(n, ast.ClassDef) and n.name == c), None)
        if node is None: break
        scope = node.body
    fn = next((n for n in ast.walk(ast.Module(body=scope, type_ignores=[]))
               if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and n.name == name), None)
    return ast.get_source_segment(source, fn) if fn is not None else None

def failing_test_text(project, bug_id, fixed_sha):
    texts = []
    for path, cls, name in test_ids(project, bug_id):
        if name is None:                                        # unittest dotted id: module.Class.test or module.test
            parts = path.split(".")
            for k in (2, 1):
                cand = "/".join(parts[:-k]) + ".py"
                src_t = file_at(project, fixed_sha, cand)
                if src_t:
                    t = extract_test(src_t, parts[-k:-1], parts[-1])
                    if t:
                        if f"# {cand}\n{t}" not in texts: texts.append(f"# {cand}\n{t}")
                        break
        else:
            src_t = file_at(project, fixed_sha, path)
            t = extract_test(src_t, cls, name) if src_t else None
            if t and f"# {path}\n{t}" not in texts: texts.append(f"# {path}\n{t}")
    return "\n\n".join(texts) if texts else None

inv = pd.read_csv(EXP1["interim"] / "bugsinpy_inventory_phase3.csv", dtype=str).set_index("sample_bug_key")
def fixed_sha_of(bug_key, project):
    ref = inv.loc[bug_key, "fixed_commit_id"].strip()
    r = git(ensure_clone(project), "rev-parse", "--verify", "-q", ref + "^{commit}")
    return r.stdout.strip() or ref

t0 = time.time(); bug_tests = {}
needed_bugs = pd.concat([eval_pairs[["sample_bug_key", "project", "bug_id"]], dev_pairs[["sample_bug_key", "project", "bug_id"]],
                         eval_unch[["sample_bug_key", "project", "bug_id"]]]).drop_duplicates("sample_bug_key")
for r in tqdm(needed_bugs.itertuples(), total=len(needed_bugs), desc="tests"):
    try:
        bug_tests[r.sample_bug_key] = failing_test_text(r.project, r.bug_id, fixed_sha_of(r.sample_bug_key, r.project))
    except Exception as e:
        bug_tests[r.sample_bug_key] = None
for d in (eval_pairs, dev_pairs, eval_unch):
    d["test"] = d.sample_bug_key.map(bug_tests)
print(f"Bugs with an extracted test: {sum(v is not None for v in bug_tests.values())} / {len(bug_tests)} "
      f"({100 * np.mean([v is not None for v in bug_tests.values()]):.1f}%) | {time.time() - t0:.0f}s")
ex = eval_pairs[eval_pairs.test.notna()].iloc[0]
print(f"\nExample test for {ex.pair_id} (first 15 lines):"); print("\n".join(ex.test.split("\n")[:15]))

# ---- Verification ----
assert np.mean([v is not None for v in bug_tests.values()]) > 0.6, "too few tests extracted"
print("\n[OK] E2-2.3")

## E2-2.4 Renaming transformation (memorisation probe)

**Explanation.** BugInPy has been public since 2020 and is probably part of the LLM's training data. If the model *recognises* a known buggy function instead of *reasoning* about it, renaming the identifiers should hurt it. We apply a **semantics-preserving, consistent renaming** with `ast`:
* parameters and local variables → `v1, v2, …` (`self` and `cls` are kept);
* in the **C0 renamed** variant, the function name is also renamed (`target_function`);
* in the **C3 renamed** variant, the function name is kept, because the test calls it.

For a pair, the **same mapping** is applied to the buggy and the fixed version, so the fix remains the only difference.

**Verification.** Every renamed function must still parse, and the number of differing lines between buggy and fixed must be unchanged by the renaming.

**Expected output.** An example (original vs renamed) and `diff preserved: 100%`.

In [ ]:
# ---- E2-2.4 Renaming ----
KEEP = {"self", "cls"}
def local_names(fn):
    names, order = set(), []
    def add(n):
        if n not in names and n not in KEEP: names.add(n); order.append(n)
    a = fn.args
    for x in a.posonlyargs + a.args + a.kwonlyargs + [a.vararg, a.kwarg]:
        if x is not None: add(x.arg)
    declared = {n for node in ast.walk(fn) if isinstance(node, (ast.Global, ast.Nonlocal)) for n in node.names}
    for node in ast.walk(fn):
        if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store): add(node.id)
        elif isinstance(node, ast.arg): add(node.arg)
    return [n for n in order if n not in declared]

class _Renamer(ast.NodeTransformer):
    def __init__(self, mapping, fn_old=None, fn_new=None): self.m, self.fo, self.fn = mapping, fn_old, fn_new
    def visit_Name(self, node):
        if node.id in self.m: node.id = self.m[node.id]
        elif self.fo and node.id == self.fo: node.id = self.fn
        return node
    def visit_arg(self, node):
        if node.arg in self.m: node.arg = self.m[node.arg]
        return node
    def visit_FunctionDef(self, node):
        if self.fo and node.name == self.fo: node.name = self.fn
        self.generic_visit(node); return node
    visit_AsyncFunctionDef = visit_FunctionDef

def rename_pair(codes, rename_function):
    fns = [parse_function(c) for c in codes]
    order = []
    for fn in fns:
        for n in local_names(fn):
            if n not in order: order.append(n)
    used = {n.id for fn in fns for n in ast.walk(fn) if isinstance(n, ast.Name)} | {fn.name for fn in fns}
    mapping, i = {}, 1
    for n in order:
        while f"v{i}" in used: i += 1
        mapping[n] = f"v{i}"; i += 1
    fn_old = fns[0].name if rename_function else None
    return [ast.unparse(_Renamer(mapping, fn_old, "target_function").visit(copy.deepcopy(fn))) for fn in fns]

def n_diff_lines(a, b):
    sm = difflib.SequenceMatcher(None, a.split("\n"), b.split("\n"), autojunk=False)
    return sum(max(i2 - i1, j2 - j1) for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal")

for d in (eval_pairs, dev_pairs):
    for rf, col in ((True, "ren_c0"), (False, "ren_c3")):
        out = [rename_pair([r.model_input_b, r.model_input_f], rf) for r in d.itertuples()]
        d[col + "_b"] = [o[0] for o in out]; d[col + "_f"] = [o[1] for o in out]
for rf, col in ((True, "ren_c0"), (False, "ren_c3")):
    eval_unch[col] = [rename_pair([c], rf)[0] for c in eval_unch.model_input]

preserved = np.mean([n_diff_lines(r.model_input_b, r.model_input_f) == n_diff_lines(r.ren_c0_b, r.ren_c0_f)
                     for r in eval_pairs.itertuples()])
ex = eval_pairs.iloc[0]
print("Original (first 8 lines):\n" + "\n".join(ex.model_input_b.split("\n")[:8]))
print("\nRenamed C0 (first 8 lines):\n" + "\n".join(ex.ren_c0_b.split("\n")[:8]))
print(f"\nDiff between buggy and fixed preserved after renaming: {100 * preserved:.1f}% of pairs")

# ---- Verification ----
for c in list(eval_pairs.ren_c0_b) + list(eval_pairs.ren_c3_f) + list(eval_unch.ren_c0): ast.parse(c)
assert preserved > 0.95, "renaming changed the buggy/fixed difference too often"
print("\n[OK] E2-2.4")

## E2-2.5 Assemble all prompts, token budgets, statistics and saving

**Explanation.** One **item** = one prompt to score. Context sections are cumulative:

| Level | Sections added |
|---|---|
| C0 | the function only |
| C1 | + *Specification (docstring)* |
| C2 | + *File context (other definitions in the same file, bodies omitted)* |
| C3 | + *Test case from the project's test suite related to this code* |

Missing context is written explicitly (*"(not available)"*), so that the prompt structure is identical for all items. **Token budgets:** function ≤ 2000, docstring ≤ 300, skeleton ≤ 1200, test ≤ 800 tokens. Each section is cut at the end if it is too long, and the cut is recorded.

| Item set | Items |
|---|---|
| evaluation, original | (2 × 150 pair members + 75 unchanged) × 4 levels |
| evaluation, renamed | the same functions at C0 and C3 |
| dev | dev pairs × 4 levels (prompt selection) |
| positive control | 100 clean functions at C0, half with `_ = 1 / 0` injected |

**Expected output.** Item counts, prompt-length statistics per level, the estimated scoring time, and the saved path.

In [ ]:
# ---- E2-2.5 Assemble items ----
BUDGET = {"code": 2000, "doc": 300, "skel": 1200, "test": 800}
TITLES = {"doc": "Specification (docstring of the function)",
          "skel": "File context (other definitions in the same file, bodies omitted)",
          "test": "Test case from the project's test suite related to this code"}
def present(x): return isinstance(x, str) and x.strip() != ""
def cut(text, key):
    if not present(text): return "(not available)", False
    ids = tok.encode(text, add_special_tokens=False)
    if len(ids) <= BUDGET[key]: return text, False
    return tok.decode(ids[:BUDGET[key]]) + "\n# ... (truncated)", True

def ladder(code, doc, skel, test):
    code_c, tr_code = cut(code, "code"); doc_c, tr_doc = cut(doc, "doc")
    skel_c, tr_skel = cut(skel, "skel"); test_c, tr_test = cut(test, "test")
    secs = [(TITLES["doc"], doc_c), (TITLES["skel"], skel_c), (TITLES["test"], test_c)]
    trunc = {"trunc_code": tr_code, "trunc_doc": tr_doc, "trunc_skel": tr_skel, "trunc_test": tr_test}
    return {f"C{k}": (code_c, secs[:k]) for k in range(4)}, trunc

items = []
def add(split, pair_id, sample_id, role, label, level, variant, project, bug, code, context, trunc, avail):
    items.append({"item_id": f"{split}|{sample_id}|{level}|{variant}", "split": split, "pair_id": pair_id,
                  "sample_id": sample_id, "role": role, "label": int(label), "level": level, "variant": variant,
                  "project": project, "sample_bug_key": bug, "code": code, "context": context, **trunc, **avail})

for split, d in (("eval", eval_pairs), ("dev", dev_pairs)):
    for r in d.itertuples():
        avail = {"has_doc": present(r.doc), "has_skel": present(r.skel), "has_test": present(r.test)}
        for role, code, sid, lab, rc0, rc3 in (("buggy", r.model_input_b, r.sample_id_b, 1, r.ren_c0_b, r.ren_c3_b),
                                               ("fixed", r.model_input_f, r.sample_id_f, 0, r.ren_c0_f, r.ren_c3_f)):
            lad, tr = ladder(code, r.doc, r.skel, r.test)
            for lv, (c, ctx) in lad.items():
                add(split, r.pair_id, sid, role, lab, lv, "original", r.project, r.sample_bug_key, c, ctx, tr, avail)
            if split == "eval":
                lad0, tr0 = ladder(rc0, r.doc, r.skel, r.test); lad3, tr3 = ladder(rc3, r.doc, r.skel, r.test)
                add(split, r.pair_id, sid, role, lab, "C0", "renamed", r.project, r.sample_bug_key, *lad0["C0"], tr0, avail)
                add(split, r.pair_id, sid, role, lab, "C3", "renamed", r.project, r.sample_bug_key, *lad3["C3"], tr3, avail)
for r in eval_unch.itertuples():
    avail = {"has_doc": present(r.doc), "has_skel": present(r.skel), "has_test": present(r.test)}
    lad, tr = ladder(r.model_input, r.doc, r.skel, r.test)
    for lv, (c, ctx) in lad.items():
        add("eval", None, r.sample_id, "unchanged", 0, lv, "original", r.project, r.sample_bug_key, c, ctx, tr, avail)
    lad0, tr0 = ladder(r.ren_c0, r.doc, r.skel, r.test); lad3, tr3 = ladder(r.ren_c3, r.doc, r.skel, r.test)
    add("eval", None, r.sample_id, "unchanged", 0, "C0", "renamed", r.project, r.sample_bug_key, *lad0["C0"], tr0, avail)
    add("eval", None, r.sample_id, "unchanged", 0, "C3", "renamed", r.project, r.sample_bug_key, *lad3["C3"], tr3, avail)
DEF_RE = re.compile(r"^(async\s+)?def\s")
def inject(code):
    lines = code.split("\n"); i = next(j for j, l in enumerate(lines) if DEF_RE.match(l))
    return "\n".join(lines[:i + 1] + ["    _ = 1 / 0"] + lines[i + 1:])
for r in pc_set.itertuples():
    code = inject(r.model_input) if r.pc_label else r.model_input
    add("pc", None, r.sample_id, "pc_injected" if r.pc_label else "pc_original", r.pc_label, "C0", "original",
        r.project, r.sample_bug_key, cut(code, "code")[0], [], {k: False for k in ("trunc_code", "trunc_doc", "trunc_skel", "trunc_test")},
        {"has_doc": False, "has_skel": False, "has_test": False})

items_df = pd.DataFrame(items)
items_df["n_tokens"] = [len(tok(build_prompt(build_user_text(c, ctx)))["input_ids"]) for c, ctx in zip(items_df.code, items_df.context)]
print("Items per split / level / variant:")
display(items_df.groupby(["split", "variant", "level"]).size().unstack("level").fillna(0).astype(int))
print("Prompt length (tokens) per level — evaluation, original:")
display(items_df[(items_df.split == "eval") & (items_df.variant == "original")].groupby("level").n_tokens
        .describe(percentiles=[.5, .9]).round(0))
print("Truncated sections (% of evaluation items):",
      {k: round(100 * items_df[items_df.split == "eval"][k].mean(), 1) for k in ("trunc_code", "trunc_doc", "trunc_skel", "trunc_test")})
sec = CONFIG2.get("scoring", {}).get("seconds_per_short_prompt", 2.0)
est_h = (items_df.n_tokens.sum() / 70 * sec) / 3600 * 0.35        # longer prompts amortise the fixed overhead
print(f"Total items: {len(items_df)} | total prompt tokens: {items_df.n_tokens.sum():,} | rough scoring time on T4: ≈ {est_h:.1f} h")

out = E2["data"] / "ladder_items.jsonl"
items_df.to_json(out, orient="records", lines=True, force_ascii=False)
CONFIG2["ladder"] = {"dev_project": DEV_PROJECT, "n_pairs": len(eval_pairs), "n_unchanged": len(eval_unch), "n_pc": len(pc_set),
                     "n_dev_pairs": len(dev_pairs), "budget_tokens": BUDGET, "titles": TITLES, "items_file": str(out),
                     "n_items": len(items_df), "items_sha256": hashlib.sha256(out.read_bytes()).hexdigest()}
save_json(CONFIG2, cfg_path)
print("Saved:", out)

# ---- Verification ----
assert items_df.item_id.is_unique
assert (items_df[items_df.split == "eval"].groupby(["pair_id", "level", "variant"]).size().dropna() == 2).all()
print("\n[OK] PHASE E2-2 COMPLETE")

**Please report back** the outputs of **E2-2.1** (counts), **E2-2.2** (coverage), **E2-2.3** (share of extracted tests), **E2-2.4** (diff preserved) and **E2-2.5** (item table, prompt lengths, estimated time).

➡️ **PHASE E2-3 (prompt selection on the dev set)** will be added after these results are confirmed.